# Initialization

In [ ]:
%pip install huggingface_hub
%pip install bgg-api
%pip install textstat

In [ ]:
import json
from huggingface_hub import InferenceClient


In [ ]:
#DRIVE
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content//NLP_proj


In [ ]:
with open('drive/MyDrive/NLP_proj/hf_api_key', 'r') as f:
    api_key = f.read()


client = InferenceClient(
    model="amd/gpt-oss-120b-chatbot",
    api_key=api_key,
)

def generate_message(sys_prompt, usr_prompt):
    return [
        {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": rulebook,
            },
    ]

# Summarize

In [ ]:
response = {}
for model in models:
    with open(f'drive/MyDrive/NLP_proj/{model}_summarize.out','r') as f:
        response[model] = json.load(f)

In [ ]:
for model in models:
    for game in file_names:
        for it in range(iterations):
            text = response[model][game][it]
            grade = textstat.flesch_kincaid_grade(text)
            print(f"Model: {model}, Game: {game}, Iteration: {it}, Grade: {grade}")

# Error Detection

In [ ]:
# READ FROM FILE
response = {}
for model in models:
    with open(f'drive/MyDrive/NLP_proj/{model}_error_detection.out','r') as f:
        response[model] = json.load(f)

In [ ]:
# EVALUATION
sys_prompt = """You are a rule‑checking assistant.
Read ANSWER and GROUND TRUTH, narrate whether the explanations refer to the same rule, and conclude with a single word **CORRECT** or **WRONG** on the final line.

### Examples
input:'''
ANSWER -> The Monopoly win condition states that the game ends when all other players have gone bankrupt,leaving a single player with all the assets. However, this rule can be problematic because it assumes that players will continue to take turns until bankruptcy occurs, which may never happen if a player repeatedly lands on “Free Parking” and collects cash without paying any fees. In such a scenario, the game could drag on indefinitely, making the win condition effectively unreachable. A more practical rule might impose a turn limit or a cash‑threshold to declare a winner when the game stalls.
GROUND TRUTH-> The problem is that the first player may reroll the dice any time.
'''

output:'''
The answer does not talk about rerolling the dice
WRONG
'''

---

input:'''
ANSWER ->  After examining the rulebook, I've identified a **gamebreaking** problem: In the first phase of the game rules states that when a player draws a card any other player can choose to draw another card. This could create a scenario where all player are willing to draw until the deck is empty.
GROUND TRUTH -> infinite card draw at the start of the game.
'''
output:'''
the answer and the ground truth adress the card draw in first phases of the game as the problem
CORRECT
'''

---

input:'''
ANSWER -> The rules appear consistent.
GROUND TRUTH -> The rulebook is consistent.
'''
'''
output:'''
both statements refer to the rule being consistent
CORRECT
'''
"""
ground_truth = {
    "ticket_to_ride": {
        "lvl0": "the rulebook is consistent",
        "lvl1": "In the rulebook it is not described how to claim a route",
        "lvl2": "In one section it is stated that during its turn a player may draw up to 2 train cards but on another line states that a player may draw up to 7 cards",
        "lvl3": "The \"a player cannot claim a route unless they have already claimed at least one other contiguous route earlier in the game\" creates a situation where a player may not clame its first route since it is not contiguous to other routes",
        "lvl4": "The rule that allows a player to reveal cards until they reveal a locomotive is unbalanced since it always allow a player to draw 2 locomotives per turn"
    },
    "dominion": {
        "lvl0": "the rulebook is consistent",
        "lvl1": "the buy phase section is missing",
        "lvl2": "the sentence \"After buying a card, a player may **continue to play additional Treasure cards** from their hand to increase their total  for that turn\" this is in direct contrast with a previous statement that does not allow a player to play treasures after the buy phase",
        "lvl3": "the sentence \"You cannot play cards that grant more action as the first action of the turn\" creates a situation where a player can't play card that grant extra action.",
        "lvl4": "the ability to play treasures also from the discard pile is unbalanced since allows player to have an ever increasing amount of gold to spend during their turn removing one of the balancing aspect of the game"
    },
    "7_wonders": {
        "lvl0": "the rulebook is consistent",
        "lvl1": "missing section explaining the resolution of military conflicts phase",
        "lvl2": "the sentence \"When buying a resource you may use Coins received from a neighbor earlier in the same turn to pay the 2‑Coin cost.\" contradicts a previous rules that states that a player cannot use coin recieved during the same turn to pay for a resource",
        "lvl3": "infinite resolve conflict loop",
        "lvl4": "the rule that allow a player to construct one stage of your Wonder for free is very unbalanced"
    },
    "catan": {
        "lvl0": "the rulebook is consistent",
        "lvl1": "missing subsection explaining building rules",
        "lvl2": "the sentence \"When you roll a 7, all hexes produce a resource except for the one with the robber.\" is directly in contrast with the rule stating that \"When you roll a 7, hexes do not produce any resources\"",
        "lvl3": "rules state that You can only trade a resource produced this turn by the hex with the robber, however since the hex with the robber does not produce resources this means that trading is not possible",
        "lvl4": "the rule stating that robber steals all resources from all players is very unbalanced since the player rolling a 7 is very likely to win the game"
    },
    "power_grid_recharged": {
        "lvl0": "the rulebook is consistent",
        "lvl1": "missing some steps in phase 5",
        "lvl2": "the sentence \"As explained in the section “The Power Plants”, each power plant may store only the number of resource tokens matching the number of symbols on the card and needs twice that number of tokens to produce electricity\" is contradicted later in the rules",
        "lvl3": "There is no way to reach phase 3 since the \"step 3\" card ir removed at the start of the game",
        "lvl4": "the rule granting a player who does not supply any city the same amount of Elektro as the highest‑earning player is unbalanced since it means that the player spending less resources is also gaining more Elektro than the others"
    }
}

evaluation_error_output = {m: {f'lvl{lvl}': {g: {
    'model_output': '',
    'ground_truth': '',
    'evaluation': '',
} }for g in file_names} for lvl in range(5)} for m in models}
for model in models:
    print(f"Model: {model}")
    for lvl in range(5):
        res = []
        # construct the answer pair
        answer_pair = []
        for game in file_names:
            for x in response[model][game][f'lvl{lvl}']:
                answer_pair.append((x, ground_truth[game][f'lvl{lvl}']))
        # check
        for answer, real in answer_pair:
            usr_prompt = f"ANSWER -> {answer}\nGROUND TRUTH -> {real}"
            out = client.chat_completion(messages=generate_message(sys_prompt, usr_prompt), temperature=0.7)
            out = out.choices[0].message.content
            # log the response
            evaluation_error_output[model][game][f'lvl{lvl}']['model_output'] = answer
            evaluation_error_output[model][game][f'lvl{lvl}']['ground_truth'] = real
            evaluation_error_output[model][game][f'lvl{lvl}']['evaluation'] = out

            if(VERBOSE):
                print(out)
            res.append(out.splitlines()[-1])
        print(f"lvl{lvl}: {res.count('CORRECT')}/{len(res)}")
with open('drive/MyDrive/NLP_proj/evaluation_error_output.json', 'w') as f:
    json.dump(evaluation_error_output, f, indent=4)

both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
The answer and the ground truth both refer to the rulebook being consistent.
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
Both statements refer to the rule being consistent
CORRECT
Both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the r

# Estimation

In [ ]:
# READ FROM FILE
response = {}
for model in models:
    with open(f'drive/MyDrive/NLP_proj/{model}_estimation.json','r') as f:
        response[model] = json.load(f)

In [ ]:
# EVALUATION
from boardgamegeek import BGGClient
bgg = BGGClient()
bgg_gt = []

for gn in ["Ticket to Ride", "Dominion", "Catan", "Power Grid Recharged"]:
    game = bgg.game(gn)
    bgg_gt.append(gn.lower().replace(" ", "_"), game.mechanics, game.complexity, game.minplayers, game.maxplayers, game.playingtime)
print(bgg_gt)

